In [1]:
# Import Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from statsmodels.tsa.seasonal import seasonal_decompose
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Set visualization style
sns.set_theme(style="whitegrid")

# Display all columns
pd.set_option("display.max_columns", None)

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
# Load dataset
df = pd.read_csv("Unemployment in India.csv")
df2 = pd.read_csv("Unemployment_Rate_upto_11_2020.csv")

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)

df.head()

Dataset loaded successfully!
Dataset shape: (768, 7)


,Region,Date,Frequency,Estimated Unemployment Rate (%),Estimated Employed,Estimated Labour Participation Rate (%),Area
0,Andhra Pradesh,31-05-2019,Monthly,3.65,11999139.0,43.24,Rural
1,Andhra Pradesh,30-06-2019,Monthly,3.05,11755881.0,42.05,Rural
2,Andhra Pradesh,31-07-2019,Monthly,3.75,12086707.0,43.50,Rural
3,Andhra Pradesh,31-08-2019,Monthly,3.32,12285693.0,43.97,Rural
4,Andhra Pradesh,30-09-2019,Monthly,5.17,12256762.0,44.68,Rural


In [3]:
# Inspect dataset
# View the first five rows
display(df.head())

,Region,Date,Frequency,Estimated Unemployment Rate (%),Estimated Employed,Estimated Labour Participation Rate (%),Area
0,Andhra Pradesh,31-05-2019,Monthly,3.65,11999139.0,43.24,Rural
1,Andhra Pradesh,30-06-2019,Monthly,3.05,11755881.0,42.05,Rural
2,Andhra Pradesh,31-07-2019,Monthly,3.75,12086707.0,43.50,Rural
3,Andhra Pradesh,31-08-2019,Monthly,3.32,12285693.0,43.97,Rural
4,Andhra Pradesh,30-09-2019,Monthly,5.17,12256762.0,44.68,Rural


In [4]:
# Check column names and data types
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 7 columns):
 #   Column                                    Non-Null Count  Dtype  
---  ------                                    --------------  -----  
 0   Region                                    740 non-null    object 
 1    Date                                     740 non-null    object 
 2    Frequency                                740 non-null    object 
 3    Estimated Unemployment Rate (%)          740 non-null    float64
 4    Estimated Employed                       740 non-null    float64
 5    Estimated Labour Participation Rate (%)  740 non-null    float64
 6   Area                                      740 non-null    object 
dtypes: float64(3), object(4)
memory usage: 42.1+ KB


In [5]:
# Check the number of rows and columns
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

Rows: 768
Columns: 7


In [6]:
# Statistical summary
display(df.describe(include="all"))

,Region,Date,Frequency,Estimated Unemployment Rate (%),Estimated Employed,Estimated Labour Participation Rate (%),Area
count,740,740,740,740.000000,7.400000e+02,740.000000,740
unique,28,14,2,NaN,NaN,NaN,2
top,Andhra Pradesh,31-10-2019,Monthly,NaN,NaN,NaN,Urban
freq,28,55,381,NaN,NaN,NaN,381
mean,NaN,NaN,NaN,11.787946,7.204460e+06,42.630122,NaN
std,NaN,NaN,NaN,10.721298,8.087988e+06,8.111094,NaN
min,NaN,NaN,NaN,0.000000,4.942000e+04,13.330000,NaN
25%,NaN,NaN,NaN,4.657500,1.190404e+06,38.062500,NaN
50%,NaN,NaN,NaN,8.350000,4.744178e+06,41.160000,NaN
75%,NaN,NaN,NaN,15.887500,1.127549e+07,45.505000,NaN


**Data cleaning**

In [7]:
# Check column names
print("Column names:")
print(df.columns.tolist())

# Check data types and non-null values
df.info()

# Count missing values in each column
print("\nMissing values:")
print(df.isnull().sum())

# Count duplicate rows
print("\nDuplicate rows:", df.duplicated().sum())

Column names:
['Region', ' Date', ' Frequency', ' Estimated Unemployment Rate (%)', ' Estimated Employed', ' Estimated Labour Participation Rate (%)', 'Area']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 7 columns):
 #   Column                                    Non-Null Count  Dtype  
---  ------                                    --------------  -----  
 0   Region                                    740 non-null    object 
 1    Date                                     740 non-null    object 
 2    Frequency                                740 non-null    object 
 3    Estimated Unemployment Rate (%)          740 non-null    float64
 4    Estimated Employed                       740 non-null    float64
 5    Estimated Labour Participation Rate (%)  740 non-null    float64
 6   Area                                      740 non-null    object 
dtypes: float64(3), object(4)
memory usage: 42.1+ KB

Missing values:
Region                      

In [8]:
# Clean column names
# Standardize column names
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
    .str.replace("-", "_", regex=False)
)

print(df.columns.tolist())

['region', 'date', 'frequency', 'estimated_unemployment_rate_(%)', 'estimated_employed', 'estimated_labour_participation_rate_(%)', 'area']


In [9]:
# Count duplicates before removal
duplicates_before = df.duplicated().sum()

print("Duplicates before cleaning:", duplicates_before)

# Remove exact duplicate rows
df = df.drop_duplicates().copy()

print("Duplicates after cleaning:", df.duplicated().sum())
print("Dataset shape after removing duplicates:", df.shape)


Duplicates before cleaning: 27
Duplicates after cleaning: 0
Dataset shape after removing duplicates: (741, 7)


In [10]:
# Missing Value
# Missing-value summary
missing_summary = pd.DataFrame({
    "Missing_Count": df.isnull().sum(),
    "Missing_Percentage": (
        df.isnull().mean() * 100
    ).round(2)
})

display(missing_summary.sort_values(
    by="Missing_Percentage",
    ascending=False
))

,Missing_Count,Missing_Percentage
region,1,0.13
date,1,0.13
frequency,1,0.13
estimated_unemployment_rate_(%),1,0.13
estimated_employed,1,0.13
estimated_labour_participation_rate_(%),1,0.13
area,1,0.13


In [12]:
df = df.dropna(
    subset=["date", "estimated_unemployment_rate_(%)"]
).copy()

In [13]:
missing_summary = pd.DataFrame({
    "Missing_Count": df.isnull().sum()
}
)

Convert the date columns

In [14]:
df["date"] = pd.to_datetime(
    df["date"],
    errors="coerce"
)

print("Date data type:", df["date"].dtype)
print("Invalid dates:", df["date"].isna().sum())

Date data type: datetime64[ns]
Invalid dates: 0


/tmp/ipykernel_8067/2990578338.py:1: UserWarning: Parsing dates in  %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df["date"] = pd.to_datetime(


In [15]:
display(df[df["date"].isna()].head(10))

,region,date,frequency,estimated_unemployment_rate_(%),estimated_employed,estimated_labour_participation_rate_(%),area


In [18]:
# Convert unemployment rates to numeric values
df["estimated_unemployment_rate_(%)"] = pd.to_numeric(
    df["estimated_unemployment_rate_(%)"]
        .astype(str)
        .str.replace("%", "", regex=False)
        .str.strip(),
    errors="coerce"
)

print(df["estimated_unemployment_rate_(%)"].dtype)
print(df["estimated_unemployment_rate_(%)"].isna().sum())

float64
0


In [19]:
# Check for invalid unemployment rates
# Inspect values outside the expected range
invalid_rates = df[
    (df["estimated_unemployment_rate_(%)"] < 0) |
    (df["estimated_unemployment_rate_(%)"] > 100)
]

print("Invalid rate records:", len(invalid_rates))
display(invalid_rates.head(10))

Invalid rate records: 0


,region,date,frequency,estimated_unemployment_rate_(%),estimated_employed,estimated_labour_participation_rate_(%),area


In [20]:
# Sort data
df = df.sort_values("date").reset_index(drop=True)

display(df.head())
display(df.tail())

,region,date,frequency,estimated_unemployment_rate_(%),estimated_employed,estimated_labour_participation_rate_(%),area
0,Andhra Pradesh,2019-05-31,Monthly,3.65,11999139.0,43.24,Rural
1,Telangana,2019-05-31,Monthly,2.23,11053353.0,61.74,Rural
2,Tamil Nadu,2019-05-31,Monthly,0.97,15844698.0,49.44,Rural
3,Rajasthan,2019-05-31,Monthly,4.03,15226005.0,38.52,Rural
4,Punjab,2019-05-31,Monthly,9.17,6088547.0,44.79,Rural


,region,date,frequency,estimated_unemployment_rate_(%),estimated_employed,estimated_labour_participation_rate_(%),area
735,Sikkim,2020-06-30,Monthly,5.81,141313.0,42.36,Rural
736,Tamil Nadu,2020-06-30,Monthly,2.06,10169115.0,31.49,Rural
737,Telangana,2020-06-30,Monthly,19.30,8891181.0,58.97,Rural
738,Himachal Pradesh,2020-06-30,Monthly,1.12,2230075.0,41.20,Rural
739,West Bengal,2020-06-30,Monthly,9.86,9088931.0,37.57,Urban


In [21]:
df = df.sort_values(
    ["region", "date"]
).reset_index(drop=True)

In [22]:
# Final data quality report
print("FINAL DATA QUALITY REPORT")
print("=" * 35)

print("Dataset shape:", df.shape)
print("Duplicate rows:", df.duplicated().sum())
print("Missing values:\n", df.isnull().sum())
print("Date data type:", df["date"].dtype)
print(
    "estimated_unemployment_rate_(%) data type:",
    df["estimated_unemployment_rate_(%)"].dtype
)

print("\nDate range:")
print("Start:", df["date"].min())
print("End:", df["date"].max())

print("\nestimated_unemployment_rate_(%):")
display(df["estimated_unemployment_rate_(%)"].describe())

FINAL DATA QUALITY REPORT
Dataset shape: (740, 7)
Duplicate rows: 0
Missing values:
 region                                     0
date                                       0
frequency                                  0
estimated_unemployment_rate_(%)            0
estimated_employed                         0
estimated_labour_participation_rate_(%)    0
area                                       0
dtype: int64
Date data type: datetime64[ns]
estimated_unemployment_rate_(%) data type: float64

Date range:
Start: 2019-05-31 00:00:00
End: 2020-06-30 00:00:00

estimated_unemployment_rate_(%):


,estimated_unemployment_rate_(%)
count,740.000000
mean,11.787946
std,10.721298
min,0.000000
25%,4.657500
50%,8.350000
75%,15.887500
max,76.740000


In [23]:
# Prepare dataset for saving
# Display current columns
print("Current columns:")
print(df.columns.tolist())

print("\nDataset shape:", df.shape)

display(df.head())

Current columns:
['region', 'date', 'frequency', 'estimated_unemployment_rate_(%)', 'estimated_employed', 'estimated_labour_participation_rate_(%)', 'area']

Dataset shape: (740, 7)


,region,date,frequency,estimated_unemployment_rate_(%),estimated_employed,estimated_labour_participation_rate_(%),area
0,Andhra Pradesh,2019-05-31,Monthly,3.65,11999139.0,43.24,Rural
1,Andhra Pradesh,2019-05-31,Monthly,6.09,4788661.0,37.45,Urban
2,Andhra Pradesh,2019-06-30,Monthly,3.80,4824630.0,36.76,Urban
3,Andhra Pradesh,2019-06-30,Monthly,3.05,11755881.0,42.05,Rural
4,Andhra Pradesh,2019-07-31,Monthly,3.75,12086707.0,43.50,Rural


In [24]:
# Save cleaned dataset
output_file = "unemployment_cleaned.csv"

df.to_csv(
    output_file,
    index=False
)

print(f"Dataset saved successfully as {output_file}")

Dataset saved successfully as unemployment_cleaned.csv


In [25]:
# Reload the saved dataset
df_check = pd.read_csv(
    "unemployment_cleaned.csv"
)

print("Saved dataset verification")
print("-" * 35)

print("Original cleaned shape:", df.shape)
print("Saved dataset shape:", df_check.shape)

print("\nMissing values in saved dataset:")
print(df_check.isnull().sum())

display(df_check.head())

Saved dataset verification
-----------------------------------
Original cleaned shape: (740, 7)
Saved dataset shape: (740, 7)

Missing values in saved dataset:
region                                     0
date                                       0
frequency                                  0
estimated_unemployment_rate_(%)            0
estimated_employed                         0
estimated_labour_participation_rate_(%)    0
area                                       0
dtype: int64


,region,date,frequency,estimated_unemployment_rate_(%),estimated_employed,estimated_labour_participation_rate_(%),area
0,Andhra Pradesh,2019-05-31,Monthly,3.65,11999139.0,43.24,Rural
1,Andhra Pradesh,2019-05-31,Monthly,6.09,4788661.0,37.45,Urban
2,Andhra Pradesh,2019-06-30,Monthly,3.80,4824630.0,36.76,Urban
3,Andhra Pradesh,2019-06-30,Monthly,3.05,11755881.0,42.05,Rural
4,Andhra Pradesh,2019-07-31,Monthly,3.75,12086707.0,43.50,Rural


In [26]:
from google.colab import files

files.download("unemployment_cleaned.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>